In [ ]:
import os
import pandas as pd
import re
import yaml  # pip install pyyaml

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\July_29\YAML_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type 1\July 29"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "3.1_YML_List_ShallowC.csv")

# === UNIT TEST KEYWORDS ===
UNIT_TEST_KEYWORDS = [
    'gradlew test', './gradlew test', 'testdebugunittest', 'testreleaseunittest',
    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

# === EXTRACT CI PLATFORM FROM FILENAME ===
def detect_ci_platform(file_path):
    filename = os.path.basename(file_path)
    if "__" in filename and "++" in filename:
        try:
            return filename.split("__")[1].split("++")[0]
        except Exception:
            return "Other"
    return "Other"

# === DETECT UNIT TEST ===
def detect_unit_test(yaml_text):
    text = yaml_text.lower()
    return any(kw in text for kw in UNIT_TEST_KEYWORDS)

# === EXTRACT ALL 'run' SCRIPTS FROM YAML ===
def extract_run_scripts(parsed):
    def get_runs(node):
        runs = []
        if isinstance(node, dict):
            for k, v in node.items():
                if k == 'run' and isinstance(v, str):
                    runs.append(v.lower())
                else:
                    runs.extend(get_runs(v))
        elif isinstance(node, list):
            for item in node:
                runs.extend(get_runs(item))
        return runs

    scripts = get_runs(parsed)
    if isinstance(parsed, dict):
        for key in ['script', 'before_script', 'before_install', 'after_script']:
            val = parsed.get(key)
            if isinstance(val, list):
                scripts.extend([str(v).lower() for v in val])
            elif isinstance(val, str):
                scripts.append(val.lower())
    return scripts

# === INSTRUMENTATION TEST DETECTION ===
def detect_instrumentation_by_platform(ci_platform, yaml_text, run_scripts, parsed):
    run_text = "\n".join(run_scripts)
    found = set()
    matched = []

    has_sdk = any(kw in run_text for kw in ['sdkmanager'])
    has_avd = any(kw in run_text for kw in ['avdmanager'])
    has_emulator = any(kw in run_text for kw in ['emulator'])
    has_create_avd = any(kw in run_text for kw in ['android create avd'])

    if ci_platform.lower() == "github_actions":
        for job in parsed.get('jobs', {}).values():
            for step in job.get('steps', []):
                if isinstance(step, dict) and 'uses' in step:
                    uses = step['uses'].lower()
                    if 'reactivecircus/android-emulator-runner' in uses:
                        found.add('GitHub_emulator_full')
                        matched.append('reactivecircus/android-emulator-runner')
                    if 'malinskiy/action-android/emulator-run-cmd' in uses:
                        found.add('GitHub_emulator_compact')
                        matched.append('malinskiy/action-android/emulator-run-cmd')
                    if any(gmd in uses for gmd in ['cleanmanageddevices', 'managedvirtualdevice', 'manageddevices']):
                        found.add('GitHub_GMD')
                        matched.append('GMD keywords (uses)')

        if any("cleanmanageddevices" in s or "manageddevices" in s for s in run_scripts):
            found.add('GitHub_GMD')
            matched.append('GMD keywords (Gradle run)')

        if (has_sdk or has_avd) and (has_emulator or has_avd):
            found.add('GitHub_emulator_manual')
            matched.append('manual emulator setup')

    elif (has_sdk or has_avd) and (has_emulator or has_avd):
        found.add(f"{ci_platform}_emulator_manual")
        matched.append('manual emulator setup')

    # === Third-party Labs ===
    for job in parsed.get('jobs', {}).values():
        for step in job.get('steps', []):
            if isinstance(step, dict) and 'uses' in step:
                uses = step['uses'].lower()
                if 'firebase-test-lab-action' in uses:
                    found.add('Firebase_Compact')
                    matched.append('Firebase-Test-Lab-Action')
                if 'microsoft/appcenter-test-cli-action' in uses:
                    found.add('Appcenter')
                    matched.append('Appcenter Test CLI Action')
                if 'browserstack' in uses:
                    found.add('Browserstack')
                    matched.append('Browserstack action')
                if 'saucelabs/saucectl-run-action' in uses:
                    found.add('SauceLabs')
                    matched.append('Sauce Labs GitHub Action')

    if 'gcloud firebase test android run' in run_text:
        found.add('Firebase_Full')
        matched.append('gcloud firebase test android run')

    if 'appcenter test run' in run_text:
        found.add('Appcenter')
        matched.append('appcenter test run')

    if 'browserstack' in run_text:
        found.add('Browserstack')
        matched.append('browserstack CLI call')

    if 'saucectl' in run_text:
        found.add('SauceLabs')
        matched.append('saucectl CLI')

    return found, matched

# === MAIN PROCESSING LOOP ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)

            if "__" in filename:
                full_name = filename.split("__")[0]
            else:
                full_name = filename

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path)

                    parsed = yaml.safe_load(raw) or {}
                    run_scripts = extract_run_scripts(parsed)

                    is_unit = detect_unit_test(raw)
                    instr_types, matched_keys = detect_instrumentation_by_platform(
                        ci_platform, raw, run_scripts, parsed
                    )

                    results.append({
                        'filename': filename,
                        'full_name': full_name,
                        'ci_platform': ci_platform,
                        'test_type': ', '.join(sorted(instr_types)),
                        'unit_test': is_unit,
                        'instrumentation_test': bool(instr_types),
                        'matched_keywords': ', '.join(sorted(set(map(str.strip, matched_keys))))
                    })

                    print(f"✅ {filename} | CI: {ci_platform} | Unit: {is_unit} | Instr: {bool(instr_types)} | Keys: {matched_keys}")

            except Exception as e:
                print(f"❌ ERROR: {file_path} => {e}")
                results.append({
                    'filename': filename,
                    'full_name': full_name,
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False,
                    'matched_keywords': ''
                })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ DONE! CSV generated at: {OUTPUT_CSV}")


✅ 0niel.university-app__GitHub_Actions++example.yaml | CI: GitHub_Actions | Unit: True | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++main.yaml | CI: GitHub_Actions | Unit: False | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++main.yml | CI: GitHub_Actions | Unit: True | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++permission_client.yaml | CI: GitHub_Actions | Unit: False | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++persistent_storage.yaml | CI: GitHub_Actions | Unit: False | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++schedule_api_client.yaml | CI: GitHub_Actions | Unit: False | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++secure_storage.yaml | CI: GitHub_Actions | Unit: False | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++shorebird-release.yml | CI: GitHub_Actions | Unit: False | Instr: False | Keys: []
✅ 0niel.university-app__GitHub_Actions++storage.